# Lab 2 · Exercise B — The on/off problem
A counting experiment with an auxiliary measurement of the background:
$n_{\rm on}=17$ events in the signal region, $n_{\rm off}=40$ in the control region, $\tau=10.53$.

$$L(s,b)=\text{Pois}(n_{\rm on}\,|\,s+b)\;\text{Pois}(n_{\rm off}\,|\,\tau b)$$

In [1]:
import ROOT
# interactive drawing in the notebook (needed on Binder)
%jsroot on
import numpy as np

MATRICOLA = 1263603
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

## B1 · Build and fit
Build the model with two `RooPoisson` and a `RooProdPdf`, import the single observation, fit, and read $\hat s$ and $\hat b$.
Hint: the means are `RooFormulaVar("mu_on", "@0+@1", ...)` and `RooFormulaVar("mu_off", "@0*@1", ...)`.

In [2]:
non, noff, tau_val = 17, 40, 10.53

s   = ROOT.RooRealVar("s", "signal", 10, 0, 200)
b   = ROOT.RooRealVar("b", "background", 4, 0.01, 200)
tau = ROOT.RooRealVar("tau", "tau", tau_val); tau.setConstant(True)
n_on  = ROOT.RooRealVar("n_on", "n_on", 0, 300)
n_off = ROOT.RooRealVar("n_off", "n_off", 0, 600)

mu_on  = ROOT.RooFormulaVar("mu_on", "@0+@1", ROOT.RooArgList(s, b))      # s + b
mu_off = ROOT.RooFormulaVar("mu_off", "@0*@1", ROOT.RooArgList(tau, b))   # tau * b
p_on   = ROOT.RooPoisson("p_on", "Pois(n_on | s+b)", n_on, mu_on)
p_off  = ROOT.RooPoisson("p_off", "Pois(n_off | tau*b)", n_off, mu_off)
pdf    = ROOT.RooProdPdf("pdf", "on x off", ROOT.RooArgList(p_on, p_off))

d = ROOT.RooDataSet("d", "d", ROOT.RooArgSet(n_on, n_off))
n_on.setVal(non); n_off.setVal(noff)
d.add(ROOT.RooArgSet(n_on, n_off))

nll   = pdf.createNLL(d)
minim = ROOT.RooMinimizer(nll)
minim.setPrintLevel(-1)
minim.minimize("Minuit2", "Migrad")
minim.hesse()
res = minim.save()
err_s_hesse = s.getError()
print("status =", res.status(), " covQual =", res.covQual(), " rho(s,b) = %+.3f" % res.correlation("s", "b"))
print("s_hat = %.2f +- %.2f   b_hat = %.2f +- %.2f" % (s.getVal(), s.getError(), b.getVal(), b.getError()))

status = 0  covQual = 3  rho(s,b) = -0.144
s_hat = 13.21 +- 4.16   b_hat = 3.80 +- 0.60


In [3]:
check("model and NLL built", pdf is not None and nll is not None and minim is not None)
check("b_hat close to n_off/tau = 3.80", abs(b.getVal() - noff / tau_val) < 0.2)
check("s_hat close to n_on - b_hat = 13.2", abs(s.getVal() - (non - noff / tau_val)) < 0.3)
check("error on b close to sqrt(n_off)/tau = 0.60", abs(b.getError() - np.sqrt(noff) / tau_val) < 0.1)

PASS  model and NLL built
PASS  b_hat close to n_off/tau = 3.80
PASS  s_hat close to n_on - b_hat = 13.2
PASS  error on b close to sqrt(n_off)/tau = 0.60


## B2 · Profile interval vs MINOS

Compute the 68 % interval on $s$ in two ways: scan the **profile** NLL until it rises by 0.5, and run **MINOS** on `s`.

*Concept Check*: MINOS *is* the numerical implementation of the profile likelihood scan. The intervals you get from the scan and from MINOS should be identical. Compare them with the symmetric HESSE errors.

`scan_interval` is provided: it needs a function returning $-\ln L_{\rm profile}(s)-(-\ln L)_{\min}$.



In [4]:
nll_min = nll.getVal()
s_hat, b_hat = s.getVal(), b.getVal()
prof = nll.createProfile(ROOT.RooArgSet(s))

def scan_interval(fn, xmax=60, n=600):
    xs = np.linspace(0.0, xmax, n)
    ys = np.array([fn(x) for x in xs])
    ok = xs[ys <= 0.5]
    return ok.min(), ok.max()

def prof_val(x):
    s.setVal(x)
    return prof.getVal()   # prof is already relative to the minimum, no "- nll_min" needed

lo_p, hi_p = scan_interval(prof_val)
print("profile : s in [%.2f, %.2f]  ->  %+.2f / %+.2f" % (lo_p, hi_p, lo_p - s_hat, hi_p - s_hat))

s.setVal(s_hat); b.setVal(b_hat)
minim.minos(ROOT.RooArgSet(s))
print("MINOS   : s = %.2f  %+.2f / %+.2f" % (s.getVal(), s.getErrorLo(), s.getErrorHi()))
print("HESSE   : s = %.2f  +- %.2f" % (s.getVal(), err_s_hesse))

profile : s in [9.42, 17.63]  ->  -3.80 / +4.41
MINOS   : s = 13.21  -3.86 / +4.49
HESSE   : s = 13.21  +- 4.16


In [5]:
check("profile interval contains the best fit", lo_p < s_hat < hi_p)
check("MINOS agrees with the profile scan (lower)", abs(s.getErrorLo() - (lo_p - s_hat)) < 0.3)
check("MINOS agrees with the profile scan (upper)", abs(s.getErrorHi() - (hi_p - s_hat)) < 0.3)

PASS  profile interval contains the best fit
PASS  MINOS agrees with the profile scan (lower)
PASS  MINOS agrees with the profile scan (upper)


In [6]:
# Plot profile
frame_prof = s.frame(ROOT.RooFit.Range(0, 40), ROOT.RooFit.Title("Profile Likelihood for s"))
# new profile object for the plot (plotting one already evaluated gives a checkForDup error)
prof_plot = nll.createProfile(ROOT.RooArgSet(s))
prof_plot.plotOn(frame_prof)
frame_prof.SetMinimum(0)
frame_prof.SetMaximum(4)
frame_prof.GetYaxis().SetTitle("#Delta(-ln L)")

# Add 0.5 line
line = ROOT.TLine(0, 0.5, 40, 0.5)
line.SetLineColor(ROOT.kGreen)
line.SetLineStyle(ROOT.kDotted)

c = ROOT.TCanvas("c", "c", 800, 500)
frame_prof.Draw()
line.Draw("SAME")
c.Draw()

### B3: The Impact of the Scale Factor $\tau$ (OPTIONAL)
The scale factor $\tau$ represents the relative exposure of the control region to the signal region. A larger $\tau$ means a more precise background estimate.
Let's see what happens if we degrade our control region. Keep the expected background $\hat{b}$ roughly the same, but change the observation to $n_{\text{off}} = 4$ and $\tau = 1.053$.

*Task*: Re-run the fit and plot the new Profile Likelihood for $s$ alongside the previous one. Observe how the interval widens.

In [7]:
# Update the RooPoisson for the off region with new values
noff_new = 4
tau_new = 1.053

# second model with its own b2, tau2 (s, n_on, n_off shared)
b2   = ROOT.RooRealVar("b2", "background", 4, 0.01, 200)
tau2 = ROOT.RooRealVar("tau2", "tau2", tau_new); tau2.setConstant(True)
mu_on2  = ROOT.RooFormulaVar("mu_on2", "@0+@1", ROOT.RooArgList(s, b2))
mu_off2 = ROOT.RooFormulaVar("mu_off2", "@0*@1", ROOT.RooArgList(tau2, b2))
p_on2   = ROOT.RooPoisson("p_on2", "p_on2", n_on, mu_on2)
p_off2  = ROOT.RooPoisson("p_off2", "p_off2", n_off, mu_off2)
pdf2    = ROOT.RooProdPdf("pdf2", "pdf2", ROOT.RooArgList(p_on2, p_off2))

d2 = ROOT.RooDataSet("d2", "d2", ROOT.RooArgSet(n_on, n_off))
n_on.setVal(non); n_off.setVal(noff_new)
d2.add(ROOT.RooArgSet(n_on, n_off))

s.setVal(10)
nll2  = pdf2.createNLL(d2)
minim2 = ROOT.RooMinimizer(nll2)
minim2.setPrintLevel(-1)
minim2.minimize("Minuit2", "Migrad")
minim2.hesse()
res2 = minim2.save()
s_hat2, b_hat2, err2_hesse = s.getVal(), b2.getVal(), s.getError()
print("tau = 1.053: status =", res2.status(), " rho(s,b) = %+.3f" % res2.correlation("s", "b2"))
print("s_hat = %.2f +- %.2f   b_hat = %.2f +- %.2f" % (s_hat2, err2_hesse, b_hat2, b2.getError()))

nll2_min = nll2.getVal()
prof2 = nll2.createProfile(ROOT.RooArgSet(s))

# slice: b fixed at the best fit, profile: b re-fitted
def slice1(x):
    s.setVal(x); b.setVal(b_hat);   return nll.getVal() - nll_min
def profile1(x):
    s.setVal(x);                    return prof.getVal()
def slice2(x):
    s.setVal(x); b2.setVal(b_hat2); return nll2.getVal() - nll2_min
def profile2(x):
    s.setVal(x);                    return prof2.getVal()

rows = []
for label, fn, sh in [("tau=10.53 slice  ", slice1, s_hat), ("tau=10.53 profile", profile1, s_hat),
                      ("tau=1.053 slice  ", slice2, s_hat2), ("tau=1.053 profile", profile2, s_hat2)]:
    lo_, hi_ = scan_interval(fn, 40, 4001)
    rows.append((label, lo_, hi_, sh))
    print("%s : s in [%.2f, %.2f]  ->  %+.2f / %+.2f   (width %.2f)" % (label, lo_, hi_, lo_ - sh, hi_ - sh, hi_ - lo_))

s.setVal(s_hat2); b2.setVal(b_hat2)
minim2.minos(ROOT.RooArgSet(s))
print("tau=1.053 MINOS   : s = %.2f  %+.2f / %+.2f   (HESSE +- %.2f)" % (s.getVal(), s.getErrorLo(), s.getErrorHi(), err2_hesse))

# both profiles on one frame
frame3 = s.frame(ROOT.RooFit.Range(0, 35), ROOT.RooFit.Title("Profile likelihood for s: good vs poor control region"))
prof_plot1 = nll.createProfile(ROOT.RooArgSet(s))
prof_plot2 = nll2.createProfile(ROOT.RooArgSet(s))
prof_plot1.plotOn(frame3, ROOT.RooFit.LineColor(ROOT.kBlue), ROOT.RooFit.Name("p1"))
prof_plot2.plotOn(frame3, ROOT.RooFit.LineColor(ROOT.kRed), ROOT.RooFit.Name("p2"))
frame3.SetMinimum(0)
frame3.SetMaximum(3)
frame3.GetYaxis().SetTitle("#Delta(-ln L)")
line3 = ROOT.TLine(0, 0.5, 35, 0.5)
line3.SetLineColor(ROOT.kGreen)
line3.SetLineStyle(ROOT.kDotted)
leg3 = ROOT.TLegend(0.55, 0.72, 0.88, 0.88)
leg3.SetTextSize(0.035)
leg3.AddEntry(frame3.findObject("p1"), "n_{off} = 40, #tau = 10.53", "l")
leg3.AddEntry(frame3.findObject("p2"), "n_{off} = 4, #tau = 1.053", "l")
c3 = ROOT.TCanvas("c3", "c3", 800, 500)
frame3.Draw()
line3.Draw("same")
leg3.Draw()
c3.Draw()

tau = 1.053: status = 0  rho(s,b) = -0.415
s_hat = 13.29 +- 4.54   b_hat = 3.79 +- 1.89
tau=10.53 slice   : s in [9.41, 17.66]  ->  -3.80 / +4.45   (width 8.25)
tau=10.53 profile : s in [9.36, 17.70]  ->  -3.85 / +4.49   (width 8.34)
tau=1.053 slice   : s in [9.42, 17.67]  ->  -3.87 / +4.38   (width 8.25)


tau=1.053 profile : s in [8.85, 17.98]  ->  -4.44 / +4.69   (width 9.13)
tau=1.053 MINOS   : s = 13.29  -4.45 / +4.70   (HESSE +- 4.54)


## Questions
Write your answer under each question, in 2–3 lines, and quote the numbers or plots of the steps indicated.

1. **(B3)** Compare the slice and the profile interval for $\tau=10.53$ and for $\tau=1.053$. Why are they almost equal in the first case and different in the second? Think of how well the control region measures `b`.
- τ = 10.53: slice [9.41, 17.66], profile [9.36, 17.70] (−3.85/+4.49, = MINOS): widths 8.25 and 8.34, almost equal. τ = 1.053: slice [9.42, 17.67] (width 8.25, unchanged), profile [8.85, 17.98] (−4.44/+4.70 with MINOS), width 9.13, 11 % wider. With n_off = 40 the control region fixes b to σ_b = √40/10.53 = 0.60 and ρ(s, b) = −0.14, so freezing b costs almost nothing. With n_off = 4, σ_b = √4/1.053 = 1.9 and ρ(s, b) = −0.42: when s moves, b can move the other way to keep s + b ≈ n_on, the profile is flatter and the interval wider. The slice ignores this, so it is too optimistic.

2. **(B1)** Which parameter or number of the model carries the uncertainty on the background? Which measurement, and which Poisson count, does it come from?
- The nuisance parameter **b** (b̂ = 3.80 ± 0.60). Its uncertainty is not an input: it comes from the control-region measurement, the factor Pois(n_off | τ b) with n_off = 40 counts, σ_b ≈ √n_off / τ. τ is a fixed, known exposure ratio and carries no uncertainty.

3. In a real analysis the control region is often a simulation or a sideband fit, not a counting measurement. Which part of the model of B1 would you change, and how?
- The auxiliary term Pois(n_off | τ b). For a simulation or an external estimate b₀ ± σ_b, I would replace it with a constraint term, e.g. Gauss(b₀ | b, σ_b) (or log-normal/Gamma to keep b > 0), with b₀ as a global observable. For a sideband fit, the sideband becomes a second channel of a simultaneous fit (`RooSimultaneous`) that shares b (and the background shape parameters) with the signal region.